# Práctica — Clase 9. Planificación y modelos

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

En esta práctica programas tú misma o tú mismo el corazón de Dyna-Q: el paso de planificación simulada. Después, mides el efecto real de variar cuántos pasos de planificación hace el agente por cada paso real.

Usa el mismo mundo-cuadrícula con muros de la clase: seis filas, nueve columnas, dos segmentos de pared, inicio en la esquina inferior izquierda y meta en la esquina superior derecha.

**Antes de empezar, ejecuta las dos celdas siguientes para tener listo el andamiaje del entorno.** No hace falta que las edites.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)


In [ ]:
# --- Mundo-cuadrícula (idéntico al de la clase) ---
ALTO, ANCHO = 6, 9
INICIO = (5, 0)
META = (0, 8)
PAREDES = {(1, 3), (2, 3), (3, 3), (0, 6), (1, 6), (2, 6)}
ACCIONES = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # arriba, abajo, izquierda, derecha
N_ACCIONES = len(ACCIONES)


def paso_entorno(estado, accion_idx):
    """Aplica una acción de forma determinista: si el movimiento choca contra
    un muro o el borde del mundo, el agente se queda donde estaba."""
    if estado == META:
        return estado, 0.0, True
    d_fila, d_col = ACCIONES[accion_idx]
    fila, col = estado
    nueva_fila = min(max(fila + d_fila, 0), ALTO - 1)
    nueva_col = min(max(col + d_col, 0), ANCHO - 1)
    nuevo_estado = (nueva_fila, nueva_col)
    if nuevo_estado in PAREDES:
        nuevo_estado = estado
    terminado = nuevo_estado == META
    recompensa = 0.0 if terminado else -1.0
    return nuevo_estado, recompensa, terminado


def indice(estado):
    """Convierte una casilla (fila, columna) en un índice único de fila en la tabla Q."""
    return estado[0] * ANCHO + estado[1]


def elegir_accion(Q, estado, epsilon):
    """Política epsilon-greedy, la misma que ya conoces de la clase tres."""
    if np.random.random() < epsilon:
        return np.random.randint(N_ACCIONES)
    fila_q = Q[indice(estado)]
    mejores = np.flatnonzero(fila_q == np.max(fila_q))
    return np.random.choice(mejores)


## Ejercicio 1 — Completa la planificación simulada de Dyna-Q

En la clase viste que Dyna-Q tiene dos fases dentro de cada paso: aprendizaje directo (con experiencia real) y planificación simulada (con experiencia imaginada por el modelo). Abajo tienes ya construida la fase uno completa, y también el `modelo`, un diccionario que guarda cada transición real que el agente ha vivido: `(estado, acción) -> (estado_siguiente, recompensa)`.

**Tu tarea es completar la fase dos, donde pone `# TODO`.** Dentro del bucle `for _ in range(n_planning)`, tienes que:

1. Elegir al azar una combinación de estado y acción que ya esté guardada en `modelo` (pista: convierte `modelo.items()` en una lista con `list(...)` y elige un índice al azar con `np.random.randint`)
2. Leer de esa entrada el estado, la acción, el estado siguiente simulado y la recompensa simulada
3. Calcular el objetivo de Q-learning para esa transición imaginada, igual que en la fase uno pero sin volver a llamar a `paso_entorno`
4. Actualizar `Q` con ese objetivo, con la misma fórmula de la fase uno

**Pistas**

- La fórmula de actualización es la misma en las dos fases: `Q[i, a] += alpha * (objetivo - Q[i, a])`
- El objetivo es `recompensa + gamma * np.max(Q[indice(estado_siguiente)])`
- No necesitas tocar el entorno real ni `elegir_accion` en esta fase: toda la información sale de `modelo`

Con el `# TODO` sin completar, la celda de prueba de abajo se ejecuta sin error, pero no notarás ninguna mejora al aumentar `n_planning`: esa es la señal de que aún te falta implementar la planificación.


In [ ]:
def ejecutar_dyna_q_incompleto(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    """Entrena un agente Dyna-Q tabular. Con n_planning=0 equivale a Q-learning puro."""
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}                      # modelo aprendido: (estado, accion) -> (estado_siguiente, recompensa)
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            # --- Fase 1: aprendizaje directo, con experiencia real (ya completa) ---
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            # --- Fase 2: planificación simulada (te toca completarla) ---
            for _ in range(n_planning):
                # TODO: 1. Elige al azar una transición ya guardada en `modelo`
                # TODO: 2. Extrae estado, acción, estado_siguiente y recompensa simulados
                # TODO: 3. Calcula el objetivo de Q-learning para esa transición imaginada
                # TODO: 4. Actualiza Q con ese objetivo, igual que en la fase 1
                pass

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio


**Ejecuta la celda siguiente para probar tu implementación.** Compara el último episodio de un agente sin planificación (`n_planning=0`) contra uno con 20 pasos de planificación (`n_planning=20`): si tu código funciona, el segundo debería tardar bastantes menos pasos en llegar a la meta.


In [ ]:
resultado_sin_planificar = ejecutar_dyna_q_incompleto(0, 30)
resultado_con_planificar = ejecutar_dyna_q_incompleto(20, 30)

print("Pasos en el último episodio, sin planificación:", resultado_sin_planificar[-1])
print("Pasos en el último episodio, con 20 pasos de planificación:", resultado_con_planificar[-1])


<details>
<summary>💡 Ver solución</summary>

```python
def ejecutar_dyna_q_incompleto(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    """Entrena un agente Dyna-Q tabular. Con n_planning=0 equivale a Q-learning puro."""
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}                      # modelo aprendido: (estado, accion) -> (estado_siguiente, recompensa)
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            # --- Fase 1: aprendizaje directo, con experiencia real ---
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            # --- Fase 2: planificación simulada ---
            transiciones_vistas = list(modelo.items())
            for _ in range(n_planning):
                # 1. Elegimos al azar una transición ya observada
                (s_sim, a_sim), (s_sig_sim, r_sim) = transiciones_vistas[
                    np.random.randint(len(transiciones_vistas))
                ]
                # 2 y 3. Calculamos el objetivo de Q-learning sobre esa transición imaginada
                i_sim, i_sig_sim = indice(s_sim), indice(s_sig_sim)
                objetivo_sim = r_sim + gamma * np.max(Q[i_sig_sim])
                # 4. Actualizamos Q con la misma fórmula que en la fase 1
                Q[i_sim, a_sim] += alpha * (objetivo_sim - Q[i_sim, a_sim])

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio
```
</details>


## Ejercicio 2 — Compara la velocidad de aprendizaje según los pasos de planificación

Ahora que tienes Dyna-Q completo, toca medir su efecto real. Vas a entrenar el agente con distintos valores de `n_planning`, promediar varias repeticiones para suavizar el azar y graficar cuántos pasos necesita cada configuración para llegar a la meta en cada episodio.

**Para que puedas avanzar aunque tu solución del ejercicio uno no esté terminada**, esta celda usa `ejecutar_dyna_q`, una copia ya completa y funcionando de la función Dyna-Q.

**Tu tarea es completar el bucle experimental**, donde pone `# TODO`:

1. Recorre cada valor de `n_planning` en `configuraciones`
2. Para cada uno, repite el entrenamiento `n_repeticiones` veces, acumulando los pasos por episodio de cada repetición
3. Guarda en `resultados[n_planning]` el promedio de esas repeticiones

**Pistas**

- Usa `np.zeros(n_episodios)` para inicializar el acumulador de cada configuración, y súmale el resultado de cada repetición
- Al final divide entre `n_repeticiones` para obtener el promedio
- La celda de gráfica ya está completa: en cuanto rellenes `resultados`, ejecútala tal cual


In [ ]:
# Copia completa de Dyna-Q, lista para usar en este ejercicio
def ejecutar_dyna_q(n_planning, n_episodios, alpha=0.1, gamma=0.95, epsilon=0.1, max_pasos=1000):
    Q = np.zeros((ALTO * ANCHO, N_ACCIONES))
    modelo = {}
    pasos_por_episodio = np.zeros(n_episodios)

    for ep in range(n_episodios):
        estado = INICIO
        pasos = 0
        terminado = False

        while not terminado and pasos < max_pasos:
            accion = elegir_accion(Q, estado, epsilon)
            sig_estado, recompensa, terminado = paso_entorno(estado, accion)

            i_estado, i_sig = indice(estado), indice(sig_estado)
            objetivo = recompensa + gamma * np.max(Q[i_sig])
            Q[i_estado, accion] += alpha * (objetivo - Q[i_estado, accion])

            modelo[(estado, accion)] = (sig_estado, recompensa)

            transiciones_vistas = list(modelo.items())
            for _ in range(n_planning):
                (s_sim, a_sim), (s_sig_sim, r_sim) = transiciones_vistas[
                    np.random.randint(len(transiciones_vistas))
                ]
                i_sim, i_sig_sim = indice(s_sim), indice(s_sig_sim)
                objetivo_sim = r_sim + gamma * np.max(Q[i_sig_sim])
                Q[i_sim, a_sim] += alpha * (objetivo_sim - Q[i_sim, a_sim])

            estado = sig_estado
            pasos += 1

        pasos_por_episodio[ep] = pasos

    return pasos_por_episodio


# --- Bucle experimental: te toca completarlo ---
n_episodios = 40
n_repeticiones = 5
configuraciones = [0, 5, 50]
resultados = {}

# TODO: recorre `configuraciones`, entrena `n_repeticiones` veces con cada
# valor de n_planning, acumula los pasos por episodio y guarda en
# `resultados[n_planning]` el promedio de las repeticiones.


In [ ]:
# --- Gráfica de la comparación (ya completa, no hace falta tocarla) ---
plt.figure(figsize=(10, 5))
for n_planning in configuraciones:
    etiqueta = "Q-learning puro (0 pasos)" if n_planning == 0 else f"Dyna-Q ({n_planning} pasos de planificación)"
    plt.plot(resultados[n_planning], label=etiqueta)
plt.xlabel("Episodio")
plt.ylabel("Pasos hasta llegar a la meta")
plt.title("Efecto del número de pasos de planificación")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Ver solución</summary>

```python
n_episodios = 40
n_repeticiones = 5
configuraciones = [0, 5, 50]
resultados = {}

for n_planning in configuraciones:
    acumulado = np.zeros(n_episodios)
    for _ in range(n_repeticiones):
        acumulado += ejecutar_dyna_q(n_planning, n_episodios)
    resultados[n_planning] = acumulado / n_repeticiones
```
</details>


## Para seguir practicando

**Cambia `configuraciones` a `[1, 10, 100]` y vuelve a ejecutar el experimento completo.** ¿A partir de cuántos pasos de planificación deja de notarse una mejora clara frente al coste extra de cómputo? Coméntalo con tu equipo.
